In [1]:
# %pip install scikit-learn
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, ElasticNet, Lasso, Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression, SelectKBest, f_regression
from scipy import stats


df = pd.read_csv('../../Datasets/fromclass/house.csv')




In [2]:


X = df.drop(['Id', 'SalePrice'], axis=1)
Y = np.log1p(df['SalePrice']) 

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)


no_feature_cols = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PoolQC', 'Fence', 'MiscFeature'
]
for col in no_feature_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna('None')
        X_test[col] = X_test[col].fillna('None')

categorical_cols = X_train.select_dtypes(include=['object']).columns
for col in categorical_cols:
        mode_val = X_train[col].mode()[0]
        X_train[col] = X_train[col].fillna(mode_val)
        X_test[col] = X_test[col].fillna(mode_val)




In [3]:
numeric_cols = X_train.select_dtypes(include=['number']).columns
for col in numeric_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)


skew_vals = X_train[numeric_cols].skew()
skewed_cols = skew_vals[skew_vals > 0.75].index
for col in skewed_cols:
    if X_train[col].nunique() > 1: 
        X_train[col] = np.log1p(X_train[col])
        X_test[col] = np.log1p(X_test[col])


X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


In [4]:
bool_cols = X_train.select_dtypes(include='bool').columns
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


selector = SelectKBest(score_func=f_regression, k=75)

X_train_selected = selector.fit_transform(X_train_scaled, Y_train)
X_test_selected = selector.transform(X_test_scaled)


lr = LinearRegression()
lr.fit(X_train_selected, Y_train)


y_pred_log = lr.predict(X_test_selected)
r2_log = r2_score(Y_test, y_pred_log)

y_pred_actual = np.expm1(y_pred_log)
y_test_actual = np.expm1(Y_test)
r2_actual = r2_score(y_test_actual, y_pred_actual)

print("------------------------------------------------")
print(f"R² (log space):   {r2_log:.4f}")
print(f"R² (actual price): {r2_actual:.4f}")
print("------------------------------------------------")


------------------------------------------------
R² (log space):   0.8820
R² (actual price): 0.8951
------------------------------------------------


In [5]:


y = np.log1p(df["SalePrice"])
X = df.drop(["SalePrice", "Id"], axis=1, errors="ignore")

# ---------------------------------
# COLUMN TYPES
# ---------------------------------
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
bool_cols = X.select_dtypes(include=["bool"]).columns.tolist()

# ---------------------------------
# BOOLEAN → INT
# ---------------------------------
for col in bool_cols:
    X[col] = X[col].astype(int)

# ---------------------------------
# NUMERIC TRANSFORMER (Skew fix + Scale)
# ---------------------------------

def skew_fix(df):
    df = df.copy()
    skew_vals = df.skew()
    skewed_cols = skew_vals[skew_vals > 0.75].index
    for col in skewed_cols:
        if df[col].nunique() > 2:
            df[col] = np.log1p(df[col])
    return df

class SkewFixTransformer:
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return skew_fix(pd.DataFrame(X)).values

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("skew_fix", SkewFixTransformer()),
    ("scaler", StandardScaler())
])

# ---------------------------------
# CATEGORICAL TRANSFORMER
# ---------------------------------
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# ---------------------------------
# FULL PREPROCESSOR
# ---------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# ---------------------------------
# FINAL PIPELINE
# ---------------------------------
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("select", SelectKBest(score_func=f_regression, k=75)),
    ("regressor", LinearRegression())
])

# ---------------------------------
# TRAIN / TEST SPLIT
# ---------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# ---------------------------------
# TRAIN MODEL
# ---------------------------------
model.fit(X_train, y_train)

# ---------------------------------
# SAVE MODEL
# ---------------------------------
joblib.dump(model, "house_price_model.pkl")
print("MODEL SAVED SUCCESSFULLY ✔")

MODEL SAVED SUCCESSFULLY ✔
